# Sample 50 Random Frame-Caption Pairs

This notebook reads `outputs/shard-00000-caption-00000.tar`, matches `.png` frames with `.txt` captions, samples 50 valid pairs, and renders them for inspection.

The notebook is structured to:
1. import the required libraries,
2. inspect the shard layout,
3. extract and match frame-caption entries,
4. sample up to 50 random pairs, and
5. display the results.

In [ ]:
import io
import random
import tarfile
from pathlib import Path

from PIL import Image
from IPython.display import Markdown, display

SHARD_PATH = Path("outputs/shard-00000-caption-00000.tar")
SAMPLE_COUNT = 50
RANDOM_SEED = 7

if not SHARD_PATH.exists():
    raise FileNotFoundError(f"Shard not found: {SHARD_PATH.resolve()}")

random.seed(RANDOM_SEED)
print(f"Using shard: {SHARD_PATH.resolve()}")
print(f"Requested sample count: {SAMPLE_COUNT}")
print(f"Random seed: {RANDOM_SEED}")

In [ ]:
with tarfile.open(SHARD_PATH, "r") as tar:
    member_names = [member.name for member in tar.getmembers()]

print(f"Total archive members: {len(member_names)}")
print("First 12 entries:")
for name in member_names[:12]:
    print(f"  {name}")

png_count = sum(name.endswith(".png") for name in member_names)
txt_count = sum(name.endswith(".txt") for name in member_names)
json_count = sum(name.endswith(".json") for name in member_names)

print()
print(f"PNG files:  {png_count}")
print(f"TXT files:  {txt_count}")
print(f"JSON files: {json_count}")

In [ ]:
frame_members = {}
caption_members = {}

for name in member_names:
    path = Path(name)
    if path.suffix == ".png":
        frame_members[path.stem] = name
    elif path.suffix == ".txt":
        caption_members[path.stem] = name

valid_keys = sorted(frame_members.keys() & caption_members.keys())
selected_count = min(SAMPLE_COUNT, len(valid_keys))
selected_keys = random.sample(valid_keys, k=selected_count) if selected_count else []

print(f"Matched frame-caption pairs: {len(valid_keys)}")
print(f"Selected pairs: {len(selected_keys)}")
if len(valid_keys) < SAMPLE_COUNT:
    print("Fewer than 50 valid pairs were available, so all pairs were selected.")

selected_keys[:10]

In [ ]:
if not selected_keys:
    raise RuntimeError("No valid frame-caption pairs were found in the shard.")

with tarfile.open(SHARD_PATH, "r") as tar:
    for index, sample_key in enumerate(selected_keys, start=1):
        frame_name = frame_members[sample_key]
        caption_name = caption_members[sample_key]

        frame_file = tar.extractfile(frame_name)
        caption_file = tar.extractfile(caption_name)
        if frame_file is None or caption_file is None:
            continue

        image = Image.open(io.BytesIO(frame_file.read())).convert("RGB")
        caption = caption_file.read().decode("utf-8").strip()

        display(Markdown(f"## Pair {index}: `{sample_key}`"))
        display(image)
        display(Markdown("**Caption**"))
        display(Markdown(f"```text\n{caption}\n```"))